In [1]:
!apt update
!apt install -y build-essential
!apt install -y mpich
!nvcc --version

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,985 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.1 MB]
Get:14 http://security.ub

In [1]:
!nvidia-smi

Mon Apr 27 16:48:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [3]:
!apt update
!apt install -y build-essential

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,546 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,863 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu

In [4]:
%%writefile a1_hello.c
#include <stdio.h>
#include <omp.h>

int main() {
    #pragma omp parallel
    {
        printf("Hello from thread %d\n", omp_get_thread_num());
    }
    return 0;
}

Writing a1_hello.c


In [5]:
!gcc a1_hello.c -fopenmp -o a1_hello
!./a1_hello

Hello from thread 0
Hello from thread 1


In [6]:
%%writefile a1_threads.c
#include <stdio.h>
#include <omp.h>

int main() {
    omp_set_num_threads(4);

    #pragma omp parallel
    {
        printf("Thread ID: %d\n", omp_get_thread_num());
    }
    return 0;
}

Writing a1_threads.c


In [7]:
!gcc a1_threads.c -fopenmp -o a1_threads
!./a1_threads

Thread ID: 3
Thread ID: 1
Thread ID: 2
Thread ID: 0


In [8]:
%%writefile a1_race.c
#include <stdio.h>
#include <omp.h>

int main() {
    int sum = 0;

    #pragma omp parallel for
    for(int i=1;i<=100;i++)
        sum += i;

    printf("Sum = %d\n", sum);
}

Writing a1_race.c


In [10]:
%%writefile a1_correct.c
#include <stdio.h>
#include <omp.h>

int main() {
    int sum = 0;

    #pragma omp parallel for reduction(+:sum)
    for(int i=1;i<=100;i++)
        sum += i;

    printf("Correct Sum = %d\n", sum);
}

Writing a1_correct.c


In [11]:
!gcc a1_correct.c -fopenmp -o a1_correct
!./a1_correct


Correct Sum = 5050


In [12]:
%%writefile a1_time.c
#include <stdio.h>
#include <omp.h>

int main() {
    long N = 100000000;
    long sum = 0;

    double start = omp_get_wtime();

    #pragma omp parallel for reduction(+:sum)
    for(long i=0;i<N;i++)
        sum += i;

    double end = omp_get_wtime();

    printf("Time taken: %f seconds\n", end-start);
}

Writing a1_time.c


In [13]:
!gcc a1_time.c -fopenmp -o a1_time
!OMP_NUM_THREADS=1 ./a1_time
!OMP_NUM_THREADS=2 ./a1_time
!OMP_NUM_THREADS=4 ./a1_time
!OMP_NUM_THREADS=8 ./a1_time

Time taken: 0.324795 seconds
Time taken: 0.137865 seconds
Time taken: 0.156456 seconds
Time taken: 0.164166 seconds


In [14]:
%%writefile a1_daxpy.c
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>

int main() {
    int N = 1<<16;
    double a = 2.5;

    double *X = malloc(N*sizeof(double));
    double *Y = malloc(N*sizeof(double));

    for(int i=0;i<N;i++){
        X[i]=1.0; Y[i]=2.0;
    }

    double start = omp_get_wtime();

    #pragma omp parallel for
    for(int i=0;i<N;i++)
        X[i] = a*X[i] + Y[i];

    double end = omp_get_wtime();

    printf("Time: %f\n", end-start);
}

Writing a1_daxpy.c


In [15]:
%%writefile a1_matrix.c
#include <stdio.h>
#include <omp.h>

#define N 500

int A[N][N], B[N][N], C[N][N];

int main() {
    for(int i=0;i<N;i++)
        for(int j=0;j<N;j++){
            A[i][j]=1;
            B[i][j]=1;
        }

    #pragma omp parallel for collapse(2)
    for(int i=0;i<N;i++)
        for(int j=0;j<N;j++){
            C[i][j]=0;
            for(int k=0;k<N;k++)
                C[i][j]+=A[i][k]*B[k][j];
        }

    printf("Done\n");
}

Writing a1_matrix.c
